# 1.05 - Haunted Places Event Type
## **Haunted Places Witness Count**

Numbers extacted using [numberscraper](https://github.com/scrapinghub/number-parser)

**"Event_Type" [str]**

- Format | 
    event_groups = {'Violence', 'Supernatural','Accident/Disaster','Flying_Object', 'Electronic_Malfunction' 'Plane_Crash'}

- Default Value: "Uknown"


## **Method**
1. Check each description for the following regex patterns using **check_regex** in *parsingFunction.py*
    - **Violence** | ["murder(?:ed|s)?"], ["kill(?:ed|s|ing)?"], etc...  

    - **Supernatural** | ["haunt(?:ed|ing|s)?"], ["poltergeist"], etc .. 

    - **Accident/Disaster** | ["drown(?:ing|ed|s)?"], ["disaster(?:s)?"], etc ..

    - **Flying Objects** | ["ufo"], ["orb" + "flying"], ["trails" + "air"], etc...

    - **Plane Crashes** | ["plane" + "crash"], etc...

    - **Electronic Malfunctions** | ["Electronic" + "jammed"], ["lights", "flickering"], etc.

2. Separate multi-part entries with "|" character

## **Notes**
- **check_regex** returns the keyword that was flagged in addition to True/False. 
    - We didn't keep this output to save space, but it is fun to look at if you run the data yourself.
- Used keywords and precompiled regex in *keywords.json*
    - keywords.events.precompiled_regex
    - keywords.events.Nouns
    - keywords.events.Descriptors
        

In [47]:
# System Path #
import os
import sys 

# Add dsci_550_a1 to base path. Lets you project functions #
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

# Pandas #
import pandas as pd
import json
import re

# Runtime #
import time
from tqdm import tqdm 

# Iterators #
import collections
import ast
import random
from typing import Pattern
from itertools import chain
from collections import Counter

from dsci_550_a1.parsingFunctions import *



## Load Data and Compile Regex

In [50]:
# Output Df
outfile = "../data/processed/haunted_places_features_added.tab"

# Reading CSV
df = pd.read_csv("../data/processed/haunted_places_cleaned.tab", sep = "\t")

# Feature Names
feature_names = ["Event_Type"]

keywords = json.load(open("../data/keywords/keywords.json"))

## Convert strings stored in "Precompiled_Regex" to regular expressions ##
for key, patterns in keywords["Precompiled_Regex"].items():
    keywords["Precompiled_Regex"][key] = [re.compile(rf"\b{pattern}\b", re.IGNORECASE) for pattern in patterns]


In [51]:
## regular expressions for nouns ##
flying_object_regex = re.compile(r"\b(" + "|".join(map(re.escape, list(chain(keywords["Nouns"]['Flying_Objects'])))) + r"s?)\b", re.IGNORECASE)
electronic_equipment_regex = re.compile(r"\b(" + "|".join(map(re.escape, list(chain(keywords["Nouns"]['Electronic_Equipment'])))) + r"s?)\b", re.IGNORECASE)
ambiguous_object_regex = re.compile(r"\b(" + "|".join(map(re.escape, list(chain(keywords["Nouns"]['Ambiguous_Objects'])))) + r"s?)\b", re.IGNORECASE)

## regular expressions for descriptors ##
malfunction_descriptor_regex = re.compile(r"\b(" + "|".join(map(re.escape, list(chain(keywords["Descriptors"]['Malfunction'])))) + r")\b", re.IGNORECASE)
crash_descriptor_regex = re.compile(r"\b(" + "|".join(map(re.escape, list(chain(keywords["Descriptors"]['Crash'])))) + r")\b", re.IGNORECASE)
flying_descriptor_regex = re.compile(r"\b(" + "|".join(map(re.escape, list(chain(keywords["Descriptors"]['Flying'])))) + r")\b", re.IGNORECASE)

## Make event_groups dictionary ##
# ["Violence", "Supernatural", "Accident/Disaster", "Flying_Object", "Electronic_Malfunction", "Plane_Crash"]
event_groups = keywords["Precompiled_Regex"]
event_groups["Flying_Object"] = [flying_object_regex, (ambiguous_object_regex, flying_descriptor_regex)]
event_groups["Electronic_Malfunction"] = [(electronic_equipment_regex, malfunction_descriptor_regex)]
event_groups["Plane_Crash"] = [(flying_object_regex, crash_descriptor_regex)]


## Feature Extraction

In [52]:
def classify_event(text, event_groups):
    if not isinstance(text, str):
        return "Unknown"

    triggered_groups = set()

    # Check every keyword in every group
    for group, patterns in event_groups.items():
        for pattern in patterns:

            if check_regex(text, pattern)[0]:
                triggered_groups.add(group)
                break

    if not triggered_groups:
        return "Unknown"

    # Return all matched groups
    return " | ".join(sorted(triggered_groups))


# Start timing
start = time.time()

# Apply classification to the "description" column.
df[f"{feature_names[0]}"] = df["Description"].apply(lambda x: classify_event(x, event_groups))

# Stop timing
end = time.time()



## Report and Save

In [53]:
# Count occurrences
counts = df[f"{feature_names[0]}"].value_counts()
unknown_counts = counts.get("Unknown", 0)

## Formatting Printout ##
extract_printout = sorted([(category, count) for category, count in counts.items()])
max_len = max(len(category) for category, _ in extract_printout)  



print("-" * 50, "Extraction Completed", "-" * 50, sep = "\n")
print(f"Extraction Took: {end - start:.6f} seconds", "-" * 50, sep = "\n")
print("\n".join([f"{category.ljust(max_len)}| {count}" for category, count in extract_printout]))
print("-" * 50)

## Save CSV ##
print("Saving to CSV...")

# Read Feature added dataframe
out_df = pd.read_csv(f"{outfile}", sep = "\t")


# Check if feature exists
for feature in feature_names:
    
    # If it exists, update values
    if feature in out_df.columns:
        out_df[feature].update(df[feature].values)
        out_df[feature] = df[feature].values

    # If not add entire column
    else:
        out_df[feature] = df[feature].values

out_df.to_csv(f"{outfile}", sep = "\t", index = False)

print(f"CSV Saved to {outfile}")

--------------------------------------------------
Extraction Completed
--------------------------------------------------
Extraction Took: 3.829719 seconds
--------------------------------------------------
Accident/Disaster                                                        | 461
Accident/Disaster | Electronic_Malfunction                               | 3
Accident/Disaster | Electronic_Malfunction | Supernatural                | 2
Accident/Disaster | Electronic_Malfunction | Supernatural | Violence     | 2
Accident/Disaster | Electronic_Malfunction | Violence                    | 5
Accident/Disaster | Flying_Object                                        | 4
Accident/Disaster | Flying_Object | Plane_Crash | Supernatural | Violence| 1
Accident/Disaster | Flying_Object | Plane_Crash | Violence               | 1
Accident/Disaster | Flying_Object | Supernatural                         | 5
Accident/Disaster | Flying_Object | Supernatural | Violence              | 1
Accident/Disaster | 